# Two-stage pipeline demo

Runs Stage 1 (Finder) + Stage 2 (Checker) on a handful of sample photos and
shows the results right here, instead of saving files to disk and reopening
them one by one.

**Heads up:** this needs a fine-tuned Stage 1 checkpoint *and* a trained Stage 2
checkpoint to actually exist. As of writing this, neither does -- Stage 1
hasn't been trained yet, and Stage 2 needs negative (non-ashwagandha) photos
that haven't been collected yet. The `YOLO(...)` load cell below will fail
until those checkpoints exist. That's expected, not a bug -- come back to this
notebook once both stages are trained.

In [ ]:
from pathlib import Path

from PIL import Image, ImageDraw
from ultralytics import YOLO
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# point these at your actual trained checkpoints once they exist
FINDER_WEIGHTS = REPO_ROOT / "runs" / "finder" / "train" / "weights" / "best.pt"
CHECKER_WEIGHTS = REPO_ROOT / "runs" / "checker" / "train" / "weights" / "best.pt"

# swap this for whatever photos you actually want to look at
SAMPLE_IMAGES = sorted((REPO_ROOT / "data" / "detect" / "images" / "test").glob("*.jpg"))[:5]

FINDER_CONF = 0.25   # kept low on purpose -- Stage 1 is meant to over-propose, Stage 2 cleans up after it
CHECKER_CONF = 0.6   # higher -- this is the stage making the final call

In [ ]:
finder = YOLO(str(FINDER_WEIGHTS))
checker = YOLO(str(CHECKER_WEIGHTS))

In [ ]:
def run_pipeline(image_path):
    image = Image.open(image_path).convert("RGB")
    finder_results = finder.predict(source=str(image_path), conf=FINDER_CONF, imgsz=1280, verbose=False)[0]
    boxes = finder_results.boxes

    draw = ImageDraw.Draw(image)
    n_candidates = 0 if boxes is None else len(boxes)
    n_confirmed = 0

    if boxes is not None:
        for xyxy, fconf in zip(boxes.xyxy.tolist(), boxes.conf.tolist()):
            x1, y1, x2, y2 = [int(v) for v in xyxy]
            crop = image.crop((x1, y1, x2, y2))
            if crop.width < 2 or crop.height < 2:
                continue  # sliver of a box, not worth checking

            checker_result = checker.predict(source=crop, verbose=False)[0]
            top1 = checker_result.probs.top1
            top1conf = float(checker_result.probs.top1conf)
            label = checker_result.names[top1]

            if label == "ashwagandha" and top1conf >= CHECKER_CONF:
                n_confirmed += 1
                draw.rectangle([x1, y1, x2, y2], outline=(0, 200, 0), width=3)
                draw.text((x1, max(0, y1 - 18)), f"ashwagandha {fconf:.2f}/{top1conf:.2f}", fill=(0, 200, 0))

    return image, n_candidates, n_confirmed

In [ ]:
for img_path in SAMPLE_IMAGES:
    annotated, n_candidates, n_confirmed = run_pipeline(img_path)
    plt.figure(figsize=(8, 6))
    plt.imshow(annotated)
    plt.title(f"{img_path.name}: {n_confirmed}/{n_candidates} boxes confirmed ashwagandha")
    plt.axis("off")
    plt.show()

### what to actually look at here

- boxes Stage 1 proposed but Stage 2 rejected (candidates minus confirmed) --
  are those correctly getting filtered out, or is Stage 2 being too strict
  and throwing away real ashwagandha leaves?
- anything Stage 1 should have boxed but just didn't -- this one's
  unrecoverable by Stage 2, it means Stage 1 itself needs more training data
  or more epochs
- if too many correct leaves are getting rejected, try lowering `CHECKER_CONF`
  and re-running